In [ ]:
# Notebook: house_prices_section2.ipynb (code cells below)
# Requirements (pip): numpy,pandas,scikit-learn,xgboost,lightgbm,catboost,matplotlib,seaborn,keras
# If on Kaggle, many of these are preinstalled.

# 1) Imports and helper functions
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

import warnings
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# --- Cross-validation with optional early stopping ---
def rmse_cv(model, X, y, n_folds=5, random_state=42, early_stopping_rounds=100):
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    rmses = []

    for train_idx, valid_idx in kf.split(X):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        fit_params = {}
        # If the model supports early stopping (XGBoost/LightGBM/CatBoost)
        if hasattr(model, "fit") and "early_stopping_rounds" in model.get_params().keys():
            fit_params = {
                "early_stopping_rounds": early_stopping_rounds,
                "eval_set": [(X_valid, y_valid)],
                "verbose": False
            }

        # Fit
        model.fit(X_train, y_train, **fit_params)

        # Predict
        preds = model.predict(X_valid)

        # Compute RMSE
        rmse = np.sqrt(mean_squared_error(y_valid, preds))
        rmses.append(rmse)

    return np.mean(rmses), np.std(rmses)


In [ ]:
# 2) Load data
train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test  = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')
train_id = train['Id']
test_id = test['Id']

# Keep target and drop Id
y = train['SalePrice']
train.drop(['Id','SalePrice'], axis=1, inplace=True)
test.drop(['Id'], axis=1, inplace=True)

In [ ]:
# 3) Quick EDA: inspect missing %
missing = pd.concat([train.isnull().mean()*100, test.isnull().mean()*100], axis=1)
missing.columns = ['train_pct_missing','test_pct_missing']
missing = missing.sort_values('train_pct_missing', ascending=False)
missing.head(20)


In [ ]:
# 4) Feature engineering strategy (apply to combined)
all_data = pd.concat([train, test], axis=0).reset_index(drop=True)
print("Combined shape:", all_data.shape)

# 4A) Impute fields with special meaning:
# Example mappings used by many notebooks: if NA means 'None' for some categorical fields (like Alley, FireplaceQu, PoolQC, Fence, MiscFeature, Garage*).
for col in ['Alley','FireplaceQu','PoolQC','Fence','MiscFeature',
            'GarageType','GarageFinish','GarageQual','GarageCond',
            'BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2',
            'MasVnrType']:
    if col in all_data.columns:
        all_data[col] = all_data[col].fillna('None')

# 4B) Numeric NA imputation: YearBuilt/YearRemod? (if any)
# Fill numeric missing values with median
num_cols = all_data.select_dtypes(include=np.number).columns.tolist()
for c in ['LotFrontage','MasVnrArea']:
    if c in all_data.columns:
        all_data[c] = all_data[c].fillna(all_data[c].median())

# For remaining numeric columns fill with 0 if appropriate (e.g. area fields where NA likely means 0)
# Use a general strategy: median for numeric, mode for categorical
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

# 4C) Create useful features
# Total square footage:
all_data['TotalSF'] = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']
# Total bathrooms (incl. basements)
all_data['TotalBath'] = (all_data['BsmtFullBath'].fillna(0) + all_data['BsmtHalfBath'].fillna(0) +
                         all_data['FullBath'].fillna(0) + 0.5 * all_data['HalfBath'].fillna(0))
# Simplify year features: house age at sale (SaleYear exists in dataset? Use YrSold)
all_data['HouseAge'] = all_data['YrSold'] - all_data['YearBuilt']
all_data['RemodAge'] = all_data['YrSold'] - all_data['YearRemodAdd']

# 4D) Convert some numeric codes that are categorical
for c in ['MSSubClass','OverallCond','OverallQual','YrSold','MoSold']:
    if c in all_data.columns:
        all_data[c] = all_data[c].astype(str)

# 4E) Log-transform the target later; transform skewed numerical predictors
# Identify numeric features to transform
numeric_feats = all_data.select_dtypes(include=[np.number]).columns
skewness = all_data[numeric_feats].apply(lambda x: x.dropna().skew()).sort_values(ascending=False)
skewed = skewness[abs(skewness) > 0.75].index
# apply log1p to reduce skewness
for feat in skewed:
    # avoid transforming indicator variables or small integer counters without positive values
    all_data[feat] = all_data[feat].apply(lambda x: np.log1p(x) if pd.notnull(x) and x>-1 else x)

# 4F) One-hot encode categorical features (for linear models and tree models without CatBoost)
categorical_feats = all_data.select_dtypes(include=['object']).columns
all_data = pd.get_dummies(all_data, columns=categorical_feats, drop_first=True)


In [ ]:
# 5) Split back
n_train = train.shape[0]
X = all_data.iloc[:n_train,:].copy()
X_test = all_data.iloc[n_train:,:].copy()

In [ ]:
# 6) Target transform
y_log = np.log1p(y)   # log transform reduces heteroscedasticity / skew

In [ ]:
# 7) Baseline models training function and model list
models = {
    "LinearRegression": make_pipeline(SimpleImputer(strategy="median"), Ridge(alpha=1.0)),
    "DecisionTree": make_pipeline(SimpleImputer(strategy="median"), DecisionTreeRegressor(max_depth=6, random_state=42)),
    "RandomForest": make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
    "GradientBoosting": make_pipeline(SimpleImputer(strategy="median"), GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42)),
    "XGBoost": xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, n_jobs=-1, random_state=42),
    "LightGBM": lgb.LGBMRegressor(n_estimators=300, learning_rate=0.03, num_leaves=31, n_jobs=-1, random_state=42),
    "CatBoost": CatBoostRegressor(verbose=0, iterations=300, learning_rate=0.03, depth=6, random_seed=42)
}


In [ ]:
# 8) Evaluate models via CV
results = []
for name, model in models.items():
    print("Evaluating:", name)
    score_mean, score_std = rmse_cv(model, X, y_log, n_folds=5)
    # convert back to RMSE in original price scale by exponent relation: rmse on log1p doesn't directly convert;
    # but for consistent ranking we can compare RMSE on log-target (common practice). We'll also show exp(RMSE) approx.
    results.append((name, score_mean, score_std))

res_df = pd.DataFrame(results, columns=['Model','CV_RMSE_log','CV_RMSE_log_std']).sort_values('CV_RMSE_log')
res_df['CV_RMSE_expapprox'] = np.expm1(res_df['CV_RMSE_log'])  # rough interpretable transform (approx)
res_df

In [ ]:
# 9) Train best model on full train and produce test predictions
best_name = res_df.iloc[0]['Model']
print("Best by CV (log rmse):", best_name)
best_model = models[best_name]
best_model.fit(X, y_log)
preds_log = best_model.predict(X_test)
preds = np.expm1(preds_log)  # back to price scale

In [ ]:
# 10) Submission
submission = pd.DataFrame({'Id': test_id, 'SalePrice': preds})
submission.to_csv('submission.csv', index=False)
print("Wrote submission.csv")

In [ ]:
# 11) Feature importance (top 10) if model supports it
def plot_feature_importance(model, X, top_n=10):
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False)[:top_n]
        plt.figure(figsize=(8,6))
        sns.barplot(x=feat_imp.values, y=feat_imp.index)
        plt.title(f"Top {top_n} Feature Importances ({type(model).__name__})")
        plt.xlabel("Importance")
        plt.tight_layout()
        plt.show()
    elif hasattr(model, 'get_feature_importance'):  # CatBoost
        importances = model.get_feature_importance()
        feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False)[:top_n]
        plt.figure(figsize=(8,6))
        sns.barplot(x=feat_imp.values, y=feat_imp.index)
        plt.title(f"Top {top_n} Feature Importances ({type(model).__name__})")
        plt.xlabel("Importance")
        plt.tight_layout()
        plt.show()
    else:
        print("Model doesn't provide feature_importances_")

plot_feature_importance(best_model, X, top_n=10)


In [ ]:

# 12) Create a neat RMSE table
display(res_df.style.format({"CV_RMSE_log": "{:.4f}", "CV_RMSE_log_std":"{:.4f}", "CV_RMSE_expapprox":"{:.2f}"}))